# FCOS Banknote Detector — Colab Training

Train an FCOS object detector for Russian ruble banknotes using a GPU runtime.

**Setup:**
1. **Runtime → Change runtime type → GPU** (T4 is free, A100/H100 with Colab Pro)
2. Fill in your HuggingFace token in Cell 6 (needed to download the dataset)
3. **Runtime → Run all**

Training keeps full 1920×1080 resolution. If the GPU runs out of memory, rent a larger GPU (H100).

Checkpoints are saved to Google Drive so training survives Colab disconnects.

In [ ]:
#@title 1. GPU Detection & Info
import subprocess, re

if not __import__('torch').cuda.is_available():
    raise RuntimeError('No GPU detected! Go to Runtime → Change runtime type → GPU')

gpu_name = __import__('torch').cuda.get_device_name(0)
gpu_mem_bytes = __import__('torch').cuda.get_device_properties(0).total_mem
gpu_mem_gb = gpu_mem_bytes / (1024**3)

print(f'GPU: {gpu_name}')
print(f'VRAM: {gpu_mem_gb:.1f} GB')

if gpu_mem_gb < 35:
    print(f'\n⚠️  WARNING: {gpu_name} has only {gpu_mem_gb:.0f} GB VRAM.')
    print('Full 1920x1080 training may OOM. Consider renting an H100 (80 GB).')
    print('Training will proceed at full resolution — if it OOMs, the process will crash.')
else:
    print('\n✅ Sufficient VRAM for full 1920x1080 training.')

In [ ]:
#@title 2. Clone Repository
import os

REPO_DIR = '/content/banknotes-detection'

if os.path.exists(REPO_DIR):
    print(f'Repository already cloned at {REPO_DIR}')
    !cd {REPO_DIR} && git pull
else:
    !git clone https://github.com/format37/banknotes-detection.git {REPO_DIR}

os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')

In [ ]:
#@title 3. Install Dependencies
!pip install -q -r requirements.txt scikit-learn

In [ ]:
#@title 4. Mount Google Drive
from google.colab import drive

drive.mount('/content/drive')

DRIVE_OUTPUT = '/content/drive/MyDrive/banknotes_training'
os.makedirs(DRIVE_OUTPUT, exist_ok=True)
print(f'Drive output directory: {DRIVE_OUTPUT}')

In [ ]:
#@title 5. HuggingFace Authentication
HF_TOKEN = ''  #@param {type:"string"}

if not HF_TOKEN:
    raise ValueError('Please enter your HuggingFace token above. '
                     'Get one at https://huggingface.co/settings/tokens')

os.environ['HF_TOKEN'] = HF_TOKEN

# Verify token works
from huggingface_hub import HfApi
user = HfApi().whoami(token=HF_TOKEN)
print(f'✅ Authenticated as: {user["name"]}')

In [ ]:
#@title 6. Write Colab Config
import json

# Load base config
with open('config.json', 'r') as f:
    config = json.load(f)

# Keep full resolution — no downscaling
# config['image_size'] is already [1920, 1080]

print('Config for training:')
print(f'  image_size:    {config["image_size"]}')
print(f'  batch_size:    {config["batch_size"]}')
print(f'  grad_accum:    {config["gradient_accumulation_steps"]}')
print(f'  epochs:        {config["epochs"]}')
print(f'  backbone:      {config["backbone_type"]}')
print(f'  checkpointing: {config.get("use_gradient_checkpointing", False)}')

# Write as separate file to keep git state clean
with open('config_colab.json', 'w') as f:
    json.dump(config, f, indent=2)

print(f'\nWritten to config_colab.json (identical to config.json — full resolution)')

In [ ]:
#@title 7. Train
!python train.py \
    --config config_colab.json \
    --output-dir outputs \
    --num-workers 2

In [ ]:
#@title 8. TensorBoard (optional — run anytime)
%load_ext tensorboard
%tensorboard --logdir outputs/

In [ ]:
#@title 9. Save Checkpoints to Google Drive
import glob, shutil

# Find the latest training run
run_dirs = sorted(glob.glob('outputs/*/'))
if not run_dirs:
    raise FileNotFoundError('No training runs found in outputs/')

latest_run = run_dirs[-1]
print(f'Latest run: {latest_run}')

# Files to copy
files_to_copy = [
    'checkpoint_best.pt',
    'checkpoint_latest.pt',
    'config.json',
]

for fname in files_to_copy:
    src = os.path.join(latest_run, fname)
    if os.path.exists(src):
        dst = os.path.join(DRIVE_OUTPUT, fname)
        print(f'Copying {src} → {dst}')
        shutil.copy2(src, dst)
    else:
        print(f'Skipping {fname} (not found)')

# Copy TensorBoard logs
logs_src = os.path.join(latest_run, 'logs')
logs_dst = os.path.join(DRIVE_OUTPUT, 'logs')
if os.path.exists(logs_src):
    if os.path.exists(logs_dst):
        shutil.rmtree(logs_dst)
    shutil.copytree(logs_src, logs_dst)
    print(f'Copied TensorBoard logs → {logs_dst}')

# Also copy any periodic checkpoints
for ckpt in glob.glob(os.path.join(latest_run, 'checkpoint_epoch_*.pt')):
    dst = os.path.join(DRIVE_OUTPUT, os.path.basename(ckpt))
    print(f'Copying {ckpt} → {dst}')
    shutil.copy2(ckpt, dst)

print(f'\n✅ All checkpoints saved to Google Drive: {DRIVE_OUTPUT}')

## Resume Training After Colab Disconnect

If Colab disconnects, re-run cells 1–6 (GPU check, clone, install, Drive, HF auth, config),
then run the cell below to resume from the latest Drive checkpoint.

In [ ]:
#@title 10. Resume Training from Drive Checkpoint
import os

DRIVE_OUTPUT = '/content/drive/MyDrive/banknotes_training'
resume_path = os.path.join(DRIVE_OUTPUT, 'checkpoint_latest.pt')

if not os.path.exists(resume_path):
    raise FileNotFoundError(f'No checkpoint found at {resume_path}. '
                            'Run training first (Cell 7) or check your Drive.')

print(f'Resuming from: {resume_path}')
!python train.py \
    --config config_colab.json \
    --output-dir outputs \
    --num-workers 2 \
    --resume {resume_path}

In [ ]:
#@title 11. Upload Best Checkpoint to HuggingFace Hub (optional)
from huggingface_hub import HfApi

DRIVE_OUTPUT = '/content/drive/MyDrive/banknotes_training'
best_ckpt = os.path.join(DRIVE_OUTPUT, 'checkpoint_best.pt')

if not os.path.exists(best_ckpt):
    raise FileNotFoundError(f'No best checkpoint at {best_ckpt}')

api = HfApi()
repo_id = 'format37/fcos-banknotes-detector'

# Create repo if it doesn't exist
api.create_repo(repo_id, exist_ok=True, token=os.environ['HF_TOKEN'])

# Upload checkpoint
api.upload_file(
    path_or_fileobj=best_ckpt,
    path_in_repo='checkpoint_best.pt',
    repo_id=repo_id,
    token=os.environ['HF_TOKEN'],
)

# Upload config
config_path = os.path.join(DRIVE_OUTPUT, 'config.json')
if os.path.exists(config_path):
    api.upload_file(
        path_or_fileobj=config_path,
        path_in_repo='config.json',
        repo_id=repo_id,
        token=os.environ['HF_TOKEN'],
    )

print(f'✅ Uploaded to https://huggingface.co/{repo_id}')